# 05 · Random Forest Training — NDSI Prediction

Train a `RandomForestRegressor` to predict NDSI from a tabular feature set: year, month, latitude, longitude, cyclone recency, and a cyclic encoding of month. The model is the engine that drives the 2050 projection in notebook 06.

**Manuscript benchmark to match (or improve on):**

| Metric | Value |
|---|---|
| Test-set R² | ~0.496 |
| Test-set RMSE (NDSI) | ~0.00539 |

**Inputs:** per-point or per-grid NDSI observations with lat/lng. This notebook expects an input CSV with at minimum `date`, `Latitude`, `Longitude`, and `ndsi`. See the note in section 1 if you only have the regional-mean series from notebook 02.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import joblib

from src.config import DATA_DIR, FIGURES_DIR, RESULTS_DIR, KFOLD_SPLITS, TEST_SIZE
from src.random_forest import (
    build_feature_matrix, train_rf, kfold_evaluate,
    feature_importance, DEFAULT_FEATURES,
)
from src.visualization import plot_feature_importance

## 1 · Load training data

**Important:** the model needs *spatial* features (lat, lng). Notebook 02's regional-mean series collapses space — for a real RF train, you need a *per-pixel* or *per-point* sampled time-series.

If you have not yet generated that, two options:

- **Quick prototype (faithful to the manuscript metrics):** sample a regular grid (e.g. 10 × 10 points across the study bbox) using `src.gee_ndsi.sample_points`, save as `data/ndsi_per_point_2000_2026.csv`, and load it here.
- **Lower-bound prototype:** treat the regional-mean series as a single 'pseudo-point' located at the bbox centroid — the model will still train but spatial features become uninformative.

The cell below loads whichever exists. Add your per-point CSV at the indicated path.

In [ ]:
per_point_path = DATA_DIR / 'ndsi_per_point_2000_2026.csv'
regional_path  = DATA_DIR / 'ndsi_timeseries_2000_2026.csv'

if per_point_path.exists():
    df = pd.read_csv(per_point_path, parse_dates=['date'])
    if 'ndsi_mean' in df.columns and 'ndsi' not in df.columns:
        df = df.rename(columns={'ndsi_mean': 'ndsi'})
    print(f'Loaded per-point training data: {len(df)} rows from {per_point_path}')
else:
    # Fallback to the regional-mean series — assigns the bbox centroid as the location.
    from src.config import STUDY_BBOX
    df = pd.read_csv(regional_path, parse_dates=['date']).rename(columns={'ndsi_mean': 'ndsi'})
    df['Latitude']  = (STUDY_BBOX[1] + STUDY_BBOX[3]) / 2
    df['Longitude'] = (STUDY_BBOX[0] + STUDY_BBOX[2]) / 2
    print(f'⚠️  Using regional-mean fallback: {len(df)} rows. ',
          f'Spatial features will be constant. Generate a per-point CSV for a faithful run.')

df.head()

## 2 · Build feature matrix

In [ ]:
X, y = build_feature_matrix(df, target_col='ndsi', date_col='date')
print(f'Feature matrix shape: {X.shape}')
print(f'Features:             {list(X.columns)}')
print(f'Target stats:         mean={y.mean():.4f}, std={y.std():.4f}, n={len(y)}')
X.head()

## 3 · K-fold cross-validation

Five folds; per-fold metrics + an aggregate row with mean ± std.

In [ ]:
kfold = kfold_evaluate(X, y, n_splits=KFOLD_SPLITS)
print(kfold.to_string(index=False))

kfold.to_csv(RESULTS_DIR / 'rf_kfold_metrics.csv', index=False)
print(f'\nSaved: {RESULTS_DIR / "rf_kfold_metrics.csv"}')

## 4 · Fit a single model on the held-out split

In [ ]:
result = train_rf(X, y, test_size=TEST_SIZE)

print(f'Train R²:     {result.r2_train:.4f}  (n_train = {result.n_train})')
print(f'Test  R²:     {result.r2_test:.4f}   (n_test  = {result.n_test})')
print(f'Test  RMSE:   {result.rmse_test:.5f}')
print(f'Test  MAE:    {result.mae_test:.5f}')

# Persist model for notebook 06
model_path = RESULTS_DIR / 'rf_ndsi_model.joblib'
joblib.dump(result.model, model_path)
print(f'\nModel persisted to: {model_path}')

## 5 · Feature importance

In [ ]:
imp = feature_importance(result.model, X.columns)
print(imp.to_string(index=False))

fig_path = FIGURES_DIR / 'fig_feature_importance.png'
plot_feature_importance(imp, save_to=fig_path)
print(f'\nSaved: {fig_path}')

---

Model is trained and persisted as `results/rf_ndsi_model.joblib`. Continue to notebook **06** for the 2050 roll-forward and hotspot mapping.